## Routines for assessing the isotropy of a 3d configuration

In [ ]:
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from analytics import *
%matplotlib widget

In [ ]:
# load data

directory = "d0.84N40k/"
output_file = directory + "bestconf.dat"

with open(directory + "/result.dat") as f:
    lines = f.readlines()
    grid_length = int(lines[4].strip().split(" ")[-1])
    jump = float(lines[3].strip().split(" ")[-1])
    
data = np.loadtxt(output_file, dtype="int")
unraveled = np.unravel_index(data, (grid_length, grid_length, grid_length))
grid = np.zeros((grid_length, grid_length, grid_length), dtype=int)
grid[unraveled] = 1

cellwidth = np.sum(grid)**(-1/3)

In [ ]:
# determine optimal inner radius of corresponding shell configuration

inner_radius = 0
if jump > r0:
    inner_radius = optimal_r_inner(jump).x[0]
outer_radius = r_outer(inner_radius)

In [ ]:
# find center of mass of data

com=np.zeros(3)

for i in range(grid_length):
    for j in range(grid_length):
        for k in range(grid_length):
            com[0]+=grid[i,j,k]*i
            com[1]+=grid[i,j,k]*j
            com[2]+=grid[i,j,k]*k
            
com = com/np.sum(grid)

print(com)

In [ ]:
# find data boundary points, following Nguyen's code

outline_lawn_points = []
lawn_points = []

for index, lawn_value in np.ndenumerate(grid):
    if lawn_value==1:
        index = np.array(index)
        lawn_points.append(index*cellwidth)
        if grid[index[0]-1,index[1],index[2]] == 0:
            outline_lawn_points.append(index*cellwidth)
        elif grid[index[0]+1,index[1],index[2]] == 0:
            outline_lawn_points.append(index*cellwidth)
        elif grid[index[0],index[1]-1,index[2]] == 0:
            outline_lawn_points.append(index*cellwidth)
        elif grid[index[0],index[1]+1,index[2]] == 0:
            outline_lawn_points.append(index*cellwidth)
        elif grid[index[0],index[1],index[2]-1] == 0:
            outline_lawn_points.append(index*cellwidth)
        elif grid[index[0],index[1],index[2]+1] == 0:
            outline_lawn_points.append(index*cellwidth)
            
lawn_points = np.array([x for x in lawn_points],dtype=float)
outline_lawn_points = np.array([x for x in outline_lawn_points],dtype=float)

center = np.array([np.mean(lawn_points[:,0]),np.mean(lawn_points[:,1]),
                   np.mean(lawn_points[:,2])])
lawn_points -= center
outline_lawn_points -= center

radii = []
for point in outline_lawn_points:
    radii.append(np.sqrt(point[0]**2+point[1]**2+point[2]**2))

In [ ]:
# plot histogram of the distance of the boundary points from the center of mass
# the vertical red lines are the inner and outer radii
# the corresponding shaded regions are the radii +/- one cell width

plt.cla()
plt.hist(radii, 60)
plt.axvline(inner_radius, color="red")
plt.axvspan(inner_radius-cellwidth, inner_radius+cellwidth, alpha=0.3, color='red')
plt.axvline(outer_radius, color="red")
plt.axvspan(outer_radius-cellwidth, outer_radius+cellwidth, alpha=0.3, color='red')
plt.show()

In [ ]:
#plot boundary

fig = plt.figure()
ax = plt.axes(projection='3d')
ax.scatter(outline_lawn_points[:,0], outline_lawn_points[:,1], 
           outline_lawn_points[:,2], c=radii[:], s=0.5)

plt.show()

In [ ]:
# layer plot of data and corresponding isotropic shape

min_layer = min(unraveled[0])
max_layer = max(unraveled[0])

fig = go.Figure()
for i in range(min_layer, max_layer + 1):
    fig.add_trace(go.Heatmap(z=grid[i]))
    
fig.add_shape(type="circle",
    xref="x", yref="y",
    x0=com[2]-outer_radius/cellwidth, y0=com[1]-outer_radius/cellwidth,
    x1=com[2]+outer_radius/cellwidth, y1=com[1]+outer_radius/cellwidth,
    line_color="LightSeaGreen",
)

fig.add_shape(type="circle",
    xref="x", yref="y",
    x0=com[2]-inner_radius/cellwidth, y0=com[1]-inner_radius/cellwidth,
    x1=com[2]+inner_radius/cellwidth, y1=com[1]+inner_radius/cellwidth,
    line_color="LightSeaGreen",
)
    
layers = []
for i in range(len(fig.data)):
    step = dict(
        method="update",
        args=[
            {"visible": [False] * len(fig.data)},
            {"title": "Layer: " + str(i)},
        ],  # layout attribute
    )
    step["args"][0]["visible"][i] = True  # Toggle i'th trace to "visible"
    layers.append(step)

sliders = [
    dict(active=0, currentvalue={"prefix": "Layer: "}, pad={"t": 50}, steps=layers)
]

fig.update_layout(sliders=sliders)

fig.update_yaxes(
    scaleanchor="x",
    scaleratio=1,
)

fig.show()

In [ ]:
# plot 3d configuration with Plotly

X, Y, Z = np.mgrid[0: grid_length, 0: grid_length, 0: grid_length]
fig = go.Figure(data=go.Volume(
    x=X.flatten(),
    y=Y.flatten(),
    z=Z.flatten(),
    value=grid.flatten(),
    isomin=0.1,
    isomax=1.0,
    opacity=0.1, # needs to be small to see through all surfaces
    surface_count=20, # needs to be a large number for good volume rendering
    ))
fig.show()

In [ ]:
# plot 3d configuration with voxels

ax = plt.figure().add_subplot(projection='3d')
ax.voxels(grid, facecolors="blue", alpha=.5) 
# You can add edgecolor="k" in the arguments to outline the individual voxels

# also plot unit sphere

u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
x = outer_radius/cellwidth*np.cos(u)*np.sin(v)+com[0]
y = outer_radius/cellwidth*np.sin(u)*np.sin(v)+com[1]
z = outer_radius/cellwidth*np.cos(v)+com[2]
ax.plot_wireframe(x, y, z, color="r")

ax.axis('auto')
plt.show()